In [2]:
import numpy as np
import math
import matplotlib.pyplot as plt
import random
from scipy.stats import poisson, norm
from scipy.optimize import least_squares




from StockPathSimulation import StockPathSimulation
from StrategySimulation import StrategySimulation

In [3]:
t = 50/252
nSims = 10
nSteps = 50

sps = StockPathSimulation(expirationTime = t,
                          numOfSims = nSims,
                          numOfSteps = nSteps)

interestRate = 0.01

ss = StrategySimulation(expirationTime = t,
                        numOfSims = nSims,
                        numOfSteps = nSteps,
                        interestRate = interestRate)

In [4]:
initialPrice = 100.0
strikePrice = np.array([80.0, 90.0, 100.0, 110.0, 120.0])

bounds = [[.1],[.7]]

trueVolatility = random.uniform(bounds[0][0], bounds[1][0])
guess = .4


process = sps.simGeomBrownianMotion(meanRateOfReturn = 0.2,
                                   volatility = trueVolatility,
                                   initialPrice = initialPrice)

optionPrice = ss.kappa(process, strikePrice, trueVolatility)



In [5]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):

    model_prices = ss.kappa(stockPrices,strikePrices, guess)

    return (marketPrices - model_prices).flatten()

result = least_squares(error_prices, x0 = guess, bounds = bounds, args = (strikePrice,process,optionPrice))

In [6]:
print(result)
print(trueVolatility)

     message: `gtol` termination condition is satisfied.
     success: True
      status: 1
         fun: [-1.421e-14 -3.411e-13 ...  0.000e+00  0.000e+00]
           x: [ 1.413e-01]
        cost: 2.9865161920995708e-22
         jac: [[-2.653e-02]
               [-3.934e+00]
               ...
               [ 0.000e+00]
               [ 0.000e+00]]
        grad: [ 6.801e-09]
  optimality: 2.81136745045009e-10
 active_mask: [0]
        nfev: 9
        njev: 9
0.14133470132027212


In [7]:
initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
error = 1e-6

bounds = [[.1, .1],[.7, 3.1]]

trueVolatility = random.uniform(bounds[0][0], bounds[1][0])
trueMeanRateOfReturn = random.uniform(.005, .1)
trueIntensity = random.uniform(1.0, 3.0)
trueLtild = trueIntensity - ((trueMeanRateOfReturn - ss.r)/trueVolatility)

guess = [.4, 1.55]

process = sps.assetModel1(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = trueIntensity,
                          initialPrice = initialPrice)

ltild = trueIntensity - ((trueMeanRateOfReturn - ss.r) / trueVolatility)

optionPrice = ss.poissonCallPrice(process, strikePrice, trueVolatility, trueLtild, error)



In [8]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility, ltild = guess
    model_prices = ss.poissonCallPrice(stockPrices, strikePrices, volatility, ltild, error)

    return (marketPrices - model_prices).flatten()

result = least_squares(error_prices, x0 = guess, bounds = bounds, args = (strikePrices, process, optionPrice))

In [9]:
print(result)
print((trueVolatility, trueLtild))

     message: `gtol` termination condition is satisfied.
     success: True
      status: 1
         fun: [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]
           x: [ 4.034e-01  1.019e+00]
        cost: 6.4398166959651536e-27
         jac: [[ 0.000e+00  0.000e+00]
               [-3.576e-07  0.000e+00]
               ...
               [ 0.000e+00  0.000e+00]
               [ 0.000e+00  0.000e+00]]
        grad: [ 7.264e-13  1.468e-13]
  optimality: 2.2043114639365962e-13
 active_mask: [0 0]
        nfev: 8
        njev: 8
(0.40344651886471683, 1.018958958710014)


In [10]:
initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
error = 1e-6

bounds = np.zeros((2,5))
bounds[0,0] = .1
bounds[1,0] = .9
for i in range(2):
    bounds[0,i+1] = -0.3
    bounds[1,i+1] = 0.3
for i in range(2):
    bounds[0,i+3] = 0
    bounds[1,i+3] = 5


trueVolatility = random.uniform(bounds[0][0], bounds[1][0])
trueMeanRateOfReturn = random.uniform(.005, .3)
trueJumps = np.random.uniform(low=-0.3, high=0.3, size= 2)
trueParameters = np.random.uniform(low=0, high=5, size= 2)
while (np.any(trueParameters) == False):
    trueParameters = np.random.uniform(low=0, high=5, size= 2)




process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = np.sum(trueParameters),
                          jumps = trueJumps,
                          jumpProbabilities = trueParameters/np.sum(trueParameters),
                          initialPrice = initialPrice)


optionPrice = ss.jumpDiffusionCallPrice(process, strikePrices, trueVolatility, trueJumps, trueParameters, error)



In [11]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility = guess[0]
    jumps = guess[1:3]
    params = guess[3:5]
    model_prices = ss.jumpDiffusionCallPrice(process, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

guess = [.5,0,0, 2.5,2.5]

result = least_squares(error_prices, x0 = guess, bounds = bounds, args = (strikePrices, process, optionPrice), max_nfev=100)

In [12]:
print(result)
print(trueVolatility)
print(trueJumps)
print(trueParameters)

     message: The maximum number of function evaluations is exceeded.
     success: False
      status: 0
         fun: [ 2.032e-04  9.399e-05 ...  0.000e+00  0.000e+00]
           x: [ 8.608e-01  9.595e-02  7.767e-04  3.162e+00  4.126e+00]
        cost: 0.0001982364651843386
         jac: [[-1.298e+01 -3.963e+00 ... -6.313e-02 -4.565e-06]
               [-1.564e+01 -4.942e+00 ... -7.784e-02 -5.605e-06]
               ...
               [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]
               [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]]
        grad: [-1.539e-01 -5.088e-02 -5.697e-04 -7.074e-04 -6.437e-08]
  optimality: 0.010382091216515137
 active_mask: [0 0 0 0 0]
        nfev: 100
        njev: 86
0.8648476015879843
[0.02928772 0.15353302]
[4.20976989 0.78714416]


In [13]:
initialPrice = 100.0
strikePrices = np.array([80.0, 90.0, 100.0, 110.0, 120.0])
error = 1e-6

bounds = np.zeros((2,7))
bounds[0,0] = .1
bounds[1,0] = .9
for i in range(3):
    bounds[0,i+1] = -0.3
    bounds[1,i+1] = 0.3
for i in range(3):
    bounds[0,i+4] = 0
    bounds[1,i+4] = 5


trueVolatility = random.uniform(bounds[0][0], bounds[1][0])
trueMeanRateOfReturn = random.uniform(.005, .3)
trueJumps = np.random.uniform(low=-0.3, high=0.3, size= 3)
trueParameters = np.random.uniform(low=0, high=5, size= 3)
while (np.any(trueParameters) == False):
    trueParameters = np.random.uniform(low=0, high=5, size= 3)




process = sps.assetModel2(meanRateOfReturn = trueMeanRateOfReturn,
                          volatility = trueVolatility,
                          intensity = np.sum(trueParameters),
                          jumps = trueJumps,
                          jumpProbabilities = trueParameters/np.sum(trueParameters),
                          initialPrice = initialPrice)


optionPrice = ss.jumpDiffusionCallPrice(process, strikePrices, trueVolatility, trueJumps, trueParameters, error)



In [14]:
def error_prices(guess, strikePrices, stockPrices, marketPrices):
    volatility = guess[0]
    jumps = guess[1:4]
    params = guess[4:7]
    model_prices = ss.jumpDiffusionCallPrice(process, strikePrices, volatility, jumps, params, error)
    
    return (marketPrices - model_prices).flatten()

guess = [.5,0,0,0, 2.5,2.5,2.5]

result = least_squares(error_prices, x0 = guess, bounds = bounds, args = (strikePrices, process, optionPrice), max_nfev=100)

In [15]:
print(result)
print(trueVolatility)
print(trueJumps)
print(trueParameters)

     message: The maximum number of function evaluations is exceeded.
     success: False
      status: 0
         fun: [ 1.723e-06 -9.473e-06 ...  0.000e+00  0.000e+00]
           x: [ 5.248e-01  2.850e-01  1.700e-01  2.125e-02  1.038e+00
                2.684e+00  1.647e+00]
        cost: 4.339764052245757e-06
         jac: [[-1.013e+01 -3.085e+00 ... -2.193e-01 -4.228e-03]
               [-1.326e+01 -4.848e+00 ... -3.120e-01 -5.600e-03]
               ...
               [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]
               [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]]
        grad: [ 1.267e-01  5.711e-02  9.897e-02  8.427e-03  8.682e-03
                3.254e-03  5.443e-05]
  optimality: 0.05383472991805391
 active_mask: [0 0 0 0 0 0 0]
        nfev: 100
        njev: 84
0.524965754142786
[0.02697235 0.17554051 0.29039596]
[1.24697569 2.73720451 0.91543563]
